# Módulo AI1 · Transformers por dentro — Práctica guiada

**Objetivo:** construir de cero las piezas de un GPT para que dejen de ser magia:

1. **Tokenizer BPE** (juguete, con un corpus de frases de TiendaSur)
2. **Atención** calculada a mano con NumPy
3. **GPT mínimo en PyTorch** entrenado en CPU (2 capas, ~70k parámetros)
4. Generar texto con el modelo entrenado

Todo corre en CPU en ~2 minutos. El ejercicio final es opcional con Hugging Face (mejor en Colab con GPU).


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print(torch.__version__)

## 1 · Corpus: frases de TiendaSur en español

Un corpus toy de 24 frases (puedes cambiarlas por datos de tu trabajo). El vocabulario que resulte será pequeño — así todo cabe en CPU.


In [ ]:
corpus = [
    "la tienda abre a las nueve",
    "la tienda cierra a las nueve",
    "el pedido llega a la tienda",
    "el cliente recoge el pedido en tienda",
    "la tienda recibe pedidos cada noche",
    "el motor del negocio es el pedido",
    "el cliente pide la factura",
    "la factura llega por correo",
    "el pedido contiene bebidas",
    "la tienda vende bebidas y comida",
    "el cliente pide bebidas frias",
    "la tienda fria vende bebidas frias",
    "el pedido llega tarde",
    "la tienda abre tarde",
    "el cliente recoge tarde",
    "el motor del pedido es el cliente",
    "la tienda recibe el correo",
    "el cliente abre la tienda",
    "el pedido llega a tiempo",
    "la factura llega a la tienda",
    "el cliente pide comida fria",
    "la tienda vende comida fria",
    "el pedido llega cada noche",
    "la tienda recibe pedidos del cliente",
]
print(len(corpus), "frases")
print("ejemplo:", corpus[0])

## 2 · Tokenizador BPE (juguete)

Entrenamos un BPE simplificado: empezamos con caracteres y fusionamos **el par más frecuente** en cada paso. Con 200 fusiones el vocabularío resultante incluye sílabas ("la ", "ped", "tie") — igual que GPT-4 con 200k fusiones, solo que en juguete.


In [ ]:
# Corpus como lista de caracteres (más un token final <eos>)
seqs = [list(s) + ["<eos>"] for s in corpus]

# Vocabulario: caracteres únicos + <eos>
vocab = sorted(set(c for s in seqs for c in s))
print("tokens iniciales:", len(vocab))

# Contar pares consecutivos en el corpus
from collections import Counter

def contar_pares(seqs):
    pares = Counter()
    for s in seqs:
        for a, b in zip(s, s[1:]):
            pares[(a, b)] += 1
    return pares

num_merges = 200
merges = []
for n in range(num_merges):
    pares = contar_pares(seqs)
    par, freq = pares.most_common(1)[0]
    if freq < 2:
        break  # nada más vale la pena fusionar
    nuevo = par[0] + par[1]
    merges.append(par)
    for i, s in enumerate(seqs):
        j = 0
        while j < len(s) - 1:
            if (s[j], s[j+1]) == par:
                s[j] = nuevo
                del s[j+1]
            j += 1

vocab = sorted(set(c for s in seqs for c in s))
print(f"tokens finales: {len(vocab)}  (con {len(merges)} fusiones BPE)")
print("algunos tokens aprendidos:", [t for t in vocab if len(t) > 1][:20])

In [ ]:
# Tokenizar de ida y vuelta (encode/decode) con el vocab y las fusiones learned
tok2id = {t: i for i, t in enumerate(vocab)}
id2tok = {i: t for t, i in tok2id.items()}

def encode_bpe(text):
    s = list(text) + ["<eos>"]
    for a, b in merges:
        j = 0
        while j < len(s) - 1:
            if (s[j], s[j+1]) == (a, b):
                s[j] = a + b
                del s[j+1]
            j += 1
    return [tok2id[t] for t in s]

def decode_bpe(ids):
    return "".join(id2tok[i] for i in ids).replace("<eos>", "")

ejemplo = encode_bpe("la tienda abre tarde")
print("IDs:", ejemplo)
print("texto:", repr(decode_bpe(ejemplo)))

## 3 · Atención a mano con NumPy (una sola head)

Tomamos una secuencia corta, la proyectamos a Q/K/V **con pesos fijos al azar** (ilustrativo), y calculamos `softmax(QK^T / sqrt(d)) @ V`.


In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # estabilidad numérica
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def atencion_causal_np(E, Wq, Wk, Wv):
    """E: (T, d) embeddings; W*: (d, d) proyecciones aprendidas."""
    T, d = E.shape
    Q, K, V = E @ Wq, E @ Wk, E @ Wv
    scores = (Q @ K.T) / math.sqrt(d)          # afinidades
    causal = np.triu(np.ones((T, T)), k=1)     # 1 sobre la diagonal
    scores = scores - 1e9 * causal             # -inf (efectivo) a futuro
    A = softmax(scores)                        # pesos que suman 1 por fila
    return A, A @ V

In [ ]:
d = 8
T = 6
E = np.random.default_rng(42).normal(size=(T, d))
Wq, Wk, Wv = (np.random.default_rng(i+1).normal(size=(d, d)) * 0.3 for i in range(3))

A, mix = atencion_causal_np(E, Wq, Wk, Wv)
print("pesos de atención (fila 0, fila 3, fila 5):")
np.set_printoptions(precision=2, suppress=True, linewidth=120)
print(A[0]); print(A[3]); print(A[5])

**Lectura de la matriz:** la fila `i` muestra **a qué tokens presta atención** el token `i`.
- El triángulo superior es 0 (máscara causal: nadie ve el futuro).
- La diagonal siempre tiene peso alto (cada token atiende a sí mismo).
- Los pesos de cada fila **suman 1**: es una mezcla de los `V` disponibles.


## 4 · GPT mínimo en PyTorch

El modelo: embeddings por token + embeddings posicionales, 2 bloques `[atención causal + MLP]` con residual y LayerNorm, y una capa que predice el siguiente token.

Es ~70k parámetros. En CPU: 200–300 pasos en <1 min.


In [ ]:
class AtencionCausal(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.dim, self.heads = dim, heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        #多头: cada head una "rebanada" del espacio
        H = self.heads
        q = q.view(B, T, H, C // H).transpose(1, 2)
        k = k.view(B, T, H, C // H).transpose(1, 2)
        v = v.view(B, T, H, C // H).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        att = att.masked_fill(mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v                       # (B,H,T,C/H)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Bloque(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.attn = AtencionCausal(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4*dim), nn.GELU(), nn.Linear(4*dim, dim))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))    # residual + atención
        x = x + self.mlp(self.ln2(x))     # residual + MLP
        return x

class GPT(nn.Module):
    def __init__(self, vocab_len, dim=64, heads=4, layers=2, ctx=32):
        super().__init__()
        self.ctx = ctx
        self.tok = nn.Embedding(vocab_len, dim)
        self.pos = nn.Embedding(ctx, dim)
        self.blocks = nn.Sequential(*[Bloque(dim, heads) for _ in range(layers)])
        self.ln = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_len)

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.blocks(x))

In [ ]:
# Datos: todos los frases del corpus como una única secuencia larga
datos = []
for s in corpus:
    datos.extend(encode_bpe(s))
datos = torch.tensor(datos, dtype=torch.long)
print("secuencia total:", len(datos), "tokens; vocab:", len(vocab))

# Batch generator: corta ventanas de ctx=32
ctx = 32
def get_batch(batch=8):
    ix = torch.randint(0, len(datos) - ctx, (batch,))
    x = torch.stack([datos[i:i+ctx] for i in ix])
    y = torch.stack([datos[i+1:i+ctx+1] for i in ix])  # desplazado: siguiente token
    return x, y

modelo = GPT(len(vocab))
opt = torch.optim.AdamW(modelo.parameters(), lr=3e-4)

paso, perdidas = 0, []
for paso in range(300):
    xb, yb = get_batch()
    logits = modelo(xb)                       # (B, T, vocab)
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.view(B*T, V), yb.view(B*T))
    opt.zero_grad(); loss.backward(); opt.step()
    perdidas.append(loss.item())
    if paso % 100 == 0:
        print(f"paso {paso:3d}  loss {loss.item():.3f}")
print(f"paso final: loss {loss.item():.3f} (ln(vocab)={math.log(len(vocab)):.2f} sería el azar)")

In [ ]:
# Curva de pérdida: debe caer del nivel del azar (ln(vocab))
import matplotlib.pyplot as plt
plt.figure(figsize=(5,3))
plt.plot(perdidas)
plt.xlabel('paso'); plt.ylabel('loss (cross-entropy)')
plt.title('GPT mínimo: aprende del azar'); plt.show()

## 5 · Generar texto con el modelo entrenado

Damos 5 palabras semilla y el modelo predice token a token. Es autoregressive: lo que genera vuelve a entrar.


In [ ]:
@torch.no_grad()
def generar(modelo, ids_semilla, max_nuevos=30, temperature=1.0):
    """Genera tokens de forma autoregresiva: contexto → logits → sample → añade."""
    ids = list(ids_semilla)
    for _ in range(max_nuevos):
        contexto = torch.tensor([ids[-modelo.ctx:]], dtype=torch.long)  # dtype explícito
        logits = modelo(contexto)[0, -1]                 # solo el último token
        probs = F.softmax(logits / temperature, dim=-1)  # temperatura: creatividad
        nuevo = torch.multinomial(probs, 1).item()
        if nuevo == tok2id["<eos>"]:
            break
        ids.append(nuevo)
    return ids

# Semilla: tokenizamos y DESCARTAMOS cualquier token que contenga <eos>
# (el BPE del juguete puede fusionar frases completas con el token <eos>).
semilla = "la tienda "
ids_semilla = [i for i in encode_bpe(semilla) if "<eos>" not in id2tok[i]]
if not ids_semilla:  # por si acaso: retrocede a los primeros caracteres
    ids_semilla = [tok2id[c] for c in list(semilla)[:2]]
print("semilla OK:", decode_bpe(ids_semilla), "→", ids_semilla)

for temp in (0.5, 1.0, 1.5):
    gen = generar(modelo, ids_semilla, max_nuevos=25, temperature=temp)
    print(f"temp={temp}: {decode_bpe(gen + [tok2id['<eos>']])!r}")

**Lectura:** con temperature baja (0.5) el modelo repite frases del corpus (más "seguro"); con alta (1.5) improvisa. Esto es exactamente el mecanismo del *decoder* de cualquier LLM en producción: sampling + temperatura (más `top_k`/`top_p`, que llegan en AI4).

---

## Ejercicios (5) — hazlo para la entrega

**1.** Con tu corpus, entrena el BPE con 400 fusiones (en lugar de 200). ¿Cuántos tokens final hay? ¿Qué frases ya se tokenizan casi en words? Comprueba `encode_bpe` sobre "el cliente recoge tarde".

**2.** Entrena el GPT con `layers=3` y `dim=128` (≈ 200k parámetros). ¿Baja el loss? ¿Qué pasa con el tiempo en CPU?

**3.** En la sección de atención manual NumPy, elimina la máscara causal (`causal = 0`). Observa la matriz y explica (2 líneas) cómo cambian los pesos: ahora todo token atiende a todos ¿qué implicación tiene para "predecir el futuro"?

**4.** Genera 10 frases con temperatura 0.25 y 10 con 1.8. Cuenta cuántas son (a) copias literales del corpus, (b) variaciones plausibles, (c) sin sentido. ¿Qué hace la temperatura, en una frase?

**5.** *(Opcional, Colab GPU)* Carga `gpt2` (124M) desde Hugging Face, genera 5 continuaciones de "la tienda abre a las" con su tokenizer y su `generate`. Compara: ¿qué cambia respecto a tu GPT de 70k? (pista: datos y escala, no el diagrama).
